
# BigQuery ML (BQML) — HR Analytics Lab
### A Plethora of Questions, Answered With SQL Alone

## Problem Statement

IndexCorp's HR team has three years of workforce data — 100 employees, 8 departments, and 1,200
monthly attendance records — sitting in BigQuery from an earlier lab. They keep asking questions
that sound like machine learning problems, but the data science team is small and every request
means exporting data to a notebook, writing Python, and shipping results back. **This notebook
answers seven distinct HR questions using nothing but SQL**, via BigQuery ML — no data ever leaves
BigQuery, and every model is trained, evaluated, and queried the same way you'd write any other
query.

## The Seven Questions (and the BQML Capability Each One Demonstrates)

| # | HR Question | BQML Capability |
|---|---|---|
| 1 | "What should this employee's salary be, given their role, department, and tenure?" | `LINEAR_REG` (regression) |
| 2 | "Which employees are likely to be high performers?" | `LOGISTIC_REG` (binary classification) |
| 3 | "Can a more powerful algorithm beat that baseline on the same question?" | `BOOSTED_TREE_CLASSIFIER` (same task, compared) |
| 4 | "Which specific factors actually drove one employee's prediction?" | `ML.FEATURE_IMPORTANCE` + `ML.EXPLAIN_PREDICT` |
| 5 | "Do employees naturally fall into distinct behavioral groups?" | `KMEANS` (unsupervised clustering) |
| 6 | "Is any employee's pattern unusual enough to flag for review?" | `ML.DETECT_ANOMALIES` (built on the KMEANS model) |
| 7 | "What should company-wide attendance/performance look like next quarter?" | `ARIMA_PLUS` (time series forecasting) |

That's regression, binary classification (two algorithms compared), explainability, clustering,
anomaly detection, and forecasting — six distinct capabilities, all in SQL, on one dataset.

## ⚠️ Cost & time notes
Every `CREATE MODEL` statement is a billed operation beyond simple query pricing (still very cheap
at this dataset's size — a few cents at most in total). None of these models need a remote
connection or external service (unlike Vector Search or remote functions from the companion
advanced notebook) — everything here trains natively inside BigQuery.

**This notebook is fully self-contained.** Authenticate → Configure → the CSVs are pulled directly
from GitHub → everything else runs against your own project.


## 1. Authenticate This Session

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")


## 2. Configure Your Project & Load the HR Dataset

In [ ]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
LOCATION = input("Enter your BigQuery location [default: US]: ").strip() or "US"
DATASET_ID = "hr_bqml_lab"

!gcloud config set project {PROJECT_ID}
!gcloud services enable bigquery.googleapis.com --project {PROJECT_ID}

from google.cloud import bigquery
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)

def TBL(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

def run_query(sql, label=None, expect_rows=True):
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            df = job.result().to_dataframe()
            display(df)
        except Exception:
            job.result()
    else:
        job.result()
    if job.total_bytes_processed is not None:
        print(f"Bytes processed: {job.total_bytes_processed:,}")
    return job

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset {DATASET_ID}")
except Conflict:
    print(f"Dataset {DATASET_ID} already exists — continuing.")


In [ ]:

GITHUB_BASE = (
    "https://raw.githubusercontent.com/himanshurathi/"
    "gcp-training-labs-accordion/master/day-1-foundations"
)

departments_df = pd.read_csv(f"{GITHUB_BASE}/departments.csv")
# parse_dates matters here: without it, pandas leaves these as plain strings, which load into
# BigQuery as STRING columns -- and DATE_DIFF() on date_of_joining (Section 3) and ARIMA_PLUS's
# time_series_timestamp_col on month_date (Question 7) both require an actual date type.
employees_df = pd.read_csv(f"{GITHUB_BASE}/employees.csv", parse_dates=["date_of_joining"])
# parse_dates alone gives a datetime64 (a timestamp), which loads into BigQuery as TIMESTAMP/
# DATETIME, not DATE -- and DATE_DIFF(CURRENT_DATE(), date_of_joining, DAY) below needs both
# sides to actually be DATE (CURRENT_DATE() returns DATE; mixing types raises a type error).
# .dt.date converts to real date objects, which load into BigQuery as true DATE.
employees_df["date_of_joining"] = employees_df["date_of_joining"].dt.date

attendance_df = pd.read_csv(f"{GITHUB_BASE}/attendance_monthly.csv", parse_dates=["month_date"])
attendance_df["month_date"] = attendance_df["month_date"].dt.date

load_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

bq_client.load_table_from_dataframe(departments_df, TBL("departments").strip("`"), job_config=load_config).result()
bq_client.load_table_from_dataframe(employees_df, TBL("employees").strip("`"), job_config=load_config).result()
bq_client.load_table_from_dataframe(attendance_df, TBL("attendance_monthly").strip("`"), job_config=load_config).result()

print("departments:", bq_client.get_table(TBL("departments").strip("`")).num_rows, "rows")
print("employees:", bq_client.get_table(TBL("employees").strip("`")).num_rows, "rows")
print("attendance_monthly:", bq_client.get_table(TBL("attendance_monthly").strip("`")).num_rows, "rows")

# confirm both date columns actually landed as date types -- catch it here, not two sections later
emp_schema = bq_client.get_table(TBL("employees").strip("`")).schema
att_schema = bq_client.get_table(TBL("attendance_monthly").strip("`")).schema
doj_type = next(f.field_type for f in emp_schema if f.name == "date_of_joining")
md_type = next(f.field_type for f in att_schema if f.name == "month_date")
print(f"\ndate_of_joining loaded as: {doj_type} | month_date loaded as: {md_type}")
assert doj_type == "DATE", (
    f"date_of_joining loaded as {doj_type}, not DATE -- DATE_DIFF(CURRENT_DATE(), ...) in "
    f"Section 3 requires both sides to be DATE, not TIMESTAMP/DATETIME"
)
assert md_type in ("DATE", "DATETIME", "TIMESTAMP"), f"month_date loaded as {md_type} -- ARIMA_PLUS in Question 7 will fail"



> 🖥️ **Check it in the console:** **BigQuery > SQL workspace > hr_bqml_lab** → all three tables
> should be listed with the row counts just printed.



## 3. Build One Reusable Feature View

**Why this matters:** every model below needs the same underlying join — employee attributes,
department name, and *aggregated* attendance behavior (an employee has 12 monthly attendance rows;
each model needs one row per employee). Building this once as a view means every `CREATE MODEL`
statement downstream is a simple `SELECT ... FROM employee_features_view`, not a repeated 15-line
join.

**The derived label:** `high_performer` is defined as an employee whose average monthly
performance rating across 2025 is 3.7 or higher — a business threshold you'd actually set with the
HR team, not something inherent in the raw data. (Worth knowing if you inspect the raw numbers:
this dataset's employee averages cluster tightly around 3.6, topping out around 3.96 — a threshold
of 4.0 would classify *zero* employees as high performers, which BQML's classifier would reject
outright as a single-class label. 3.7 gives a real ~20/80 split.)


In [ ]:

run_query(f'''
CREATE OR REPLACE VIEW {TBL("employee_features_view")} AS
SELECT
  e.employee_id,
  e.gender,
  e.state,
  d.department_name,
  e.designation,
  DATE_DIFF(CURRENT_DATE(), e.date_of_joining, DAY) AS tenure_days,
  e.monthly_salary_inr,
  ROUND(AVG(a.days_present), 2) AS avg_days_present,
  ROUND(AVG(a.leaves_taken), 2) AS avg_leaves_taken,
  ROUND(AVG(a.performance_rating), 2) AS avg_performance_rating,
  (AVG(a.performance_rating) >= 3.7) AS high_performer
FROM {TBL("employees")} e
JOIN {TBL("departments")} d ON e.department_id = d.department_id
JOIN {TBL("attendance_monthly")} a ON e.employee_id = a.employee_id
GROUP BY e.employee_id, e.gender, e.state, d.department_name, e.designation,
         e.date_of_joining, e.monthly_salary_inr
''', label="Create employee_features_view", expect_rows=False)

run_query(f'''
SELECT * FROM {TBL("employee_features_view")} LIMIT 10
''', label="Preview the feature view")



> 🖥️ **Check it in the console:** **hr_bqml_lab > employee_features_view** → Details tab shows
> **Table type: VIEW**. Every model in this notebook selects from this one view — a good moment to
> point out that "feature engineering" here is just a `GROUP BY` and a derived boolean column.



## Question 1 — "What should this employee's salary be?" (Linear Regression)

**The business problem:** compensation benchmarking — is a given employee's salary in line with
what their role, department, tenure, and location would predict, or is it a outlier worth
reviewing?

**Why `linear_reg`:** the label (`monthly_salary_inr`) is a continuous number, not a category —
regression, not classification.


In [ ]:

run_query(f'''
CREATE OR REPLACE MODEL {TBL("salary_model")}
OPTIONS(model_type='linear_reg', input_label_cols=['monthly_salary_inr']) AS
SELECT department_name, designation, state, gender, tenure_days, monthly_salary_inr
FROM {TBL("employee_features_view")}
''', label="Train salary_model", expect_rows=False)

run_query(f'''
SELECT * FROM ML.EVALUATE(MODEL {TBL("salary_model")})
''', label="Evaluate salary_model — look at r2_score and mean_absolute_error")


In [ ]:

run_query(f'''
SELECT
  employee_id, department_name, designation, monthly_salary_inr AS actual_salary,
  ROUND(predicted_monthly_salary_inr, 2) AS predicted_salary,
  ROUND(monthly_salary_inr - predicted_monthly_salary_inr, 2) AS difference
FROM ML.PREDICT(MODEL {TBL("salary_model")}, (
  SELECT employee_id, department_name, designation, state, gender, tenure_days, monthly_salary_inr
  FROM {TBL("employee_features_view")}
))
ORDER BY ABS(difference) DESC
LIMIT 10
''', label="Largest actual-vs-predicted salary gaps")



**What to look for:** the employees with the largest gap between actual and predicted salary are
exactly the ones a compensation review would want to look at first — either genuinely
under/overpaid relative to peers, or the model is missing a real factor (e.g. a special skill)
worth discussing with HR.

> 🖥️ **Check it in the console:** **hr_bqml_lab > salary_model** → **Evaluation** tab shows
> `r2_score` and `mean_absolute_error` as summary cards, and **Training** tab shows the loss curve
> across iterations — both computed automatically.



## Question 2 — "Which employees are likely to be high performers?" (Logistic Regression)

**The business problem:** proactively identify high performers for retention/promotion
conversations, rather than only noticing after an exit interview.

**Why `logistic_reg`:** the label (`high_performer`) is binary (true/false) — classification, not
regression.


In [ ]:

run_query(f'''
CREATE OR REPLACE MODEL {TBL("high_performer_logistic_model")}
OPTIONS(model_type='logistic_reg', input_label_cols=['high_performer']) AS
SELECT department_name, designation, state, gender, tenure_days,
       avg_days_present, avg_leaves_taken, high_performer
FROM {TBL("employee_features_view")}
''', label="Train high_performer_logistic_model", expect_rows=False)

run_query(f'''
SELECT * FROM ML.EVALUATE(MODEL {TBL("high_performer_logistic_model")})
''', label="Evaluate — look at precision, recall, roc_auc")



> 🖥️ **Check it in the console:** **hr_bqml_lab > high_performer_logistic_model > Evaluation** tab
> → precision/recall and an ROC curve are rendered visually — note these are *different* metric
> columns than Question 1's `r2_score`, since `ML.EVALUATE`'s output shape depends on model type.



## Question 3 — Can a More Powerful Algorithm Beat That Baseline? (Boosted Tree)

**The business problem:** logistic regression is fast and interpretable, but is it actually the
*best* model for this prediction, or just the simplest? Training a second, more flexible algorithm
on the **exact same features and label** gives a fair, direct comparison.


In [ ]:

run_query(f'''
CREATE OR REPLACE MODEL {TBL("high_performer_boosted_tree_model")}
OPTIONS(model_type='boosted_tree_classifier', input_label_cols=['high_performer']) AS
SELECT department_name, designation, state, gender, tenure_days,
       avg_days_present, avg_leaves_taken, high_performer
FROM {TBL("employee_features_view")}
''', label="Train high_performer_boosted_tree_model", expect_rows=False)

run_query(f'''
SELECT * FROM ML.EVALUATE(MODEL {TBL("high_performer_boosted_tree_model")})
''', label="Evaluate the boosted tree model")


In [ ]:

print("Side-by-side comparison — same task, two algorithms:\n")
logistic_eval = bq_client.query(
    f"SELECT 'logistic_reg' AS model, * FROM ML.EVALUATE(MODEL {TBL('high_performer_logistic_model')})"
).result().to_dataframe()
boosted_eval = bq_client.query(
    f"SELECT 'boosted_tree' AS model, * FROM ML.EVALUATE(MODEL {TBL('high_performer_boosted_tree_model')})"
).result().to_dataframe()

comparison = pd.concat([logistic_eval, boosted_eval], ignore_index=True)
display(comparison[["model", "precision", "recall", "roc_auc"]])



**What to look for:** on a dataset this small (100 employees), the two models will often perform
similarly, or the simpler logistic regression may even win — a genuinely useful, realistic lesson:
**more powerful ≠ automatically better**, especially on small data where a flexible model can
overfit. At real production scale (thousands+ of employees), the boosted tree typically pulls
ahead by capturing non-linear interactions logistic regression can't.

> 🖥️ **Check it in the console:** **hr_bqml_lab** → both models listed side by side → open each
> one's **Evaluation** tab in separate browser tabs for a visual comparison matching the printed
> table above.



## Question 4 — Which Factors Actually Drove a Prediction? (Explainability)

**The business problem:** a manager asks "why did the model flag this specific employee as a
high performer?" — a raw prediction score alone doesn't answer that; explainability does.

### 4.1 Global Feature Importance (Boosted Tree Models Only)
**What this shows:** across *all* employees, which features the model relied on most overall.


In [ ]:

run_query(f'''
SELECT * FROM ML.FEATURE_IMPORTANCE(MODEL {TBL("high_performer_boosted_tree_model")})
ORDER BY importance_gain DESC
''', label="Global feature importance")



### 4.2 Per-Prediction Explanation
**What this shows:** for one *specific* employee, exactly how much each feature pushed the
prediction toward or away from "high performer."


In [ ]:

run_query(f'''
SELECT
  employee_id, predicted_high_performer, top_feature_attributions
FROM ML.EXPLAIN_PREDICT(MODEL {TBL("high_performer_boosted_tree_model")}, (
  SELECT employee_id, department_name, designation, state, gender, tenure_days,
         avg_days_present, avg_leaves_taken
  FROM {TBL("employee_features_view")}
  LIMIT 5
), STRUCT(3 AS top_k_features))
''', label="Per-employee explanation (top 3 contributing features each)")



**What to look for:** `top_feature_attributions` lists each employee's top contributing features
with signed attribution values — a positive attribution pushed the prediction toward "high
performer," negative pushed away. This is the concrete answer to "why did the model say that,"
not just a probability score.

> 🖥️ **Check it in the console:** no dedicated console visualization for per-prediction
> explanations — this is a query-only capability, best explored directly in the SQL workspace
> results grid.



## Question 5 — Do Employees Naturally Fall Into Distinct Groups? (K-Means)

**The business problem:** rather than guessing at employee segments, let unsupervised learning
find natural groupings based on attendance behavior — no label needed, since there's no
predetermined "correct" segment.


In [ ]:

run_query(f'''
CREATE OR REPLACE MODEL {TBL("employee_segments_model")}
OPTIONS(model_type='kmeans', num_clusters=3) AS
SELECT avg_days_present, avg_leaves_taken, avg_performance_rating
FROM {TBL("employee_features_view")}
''', label="Train employee_segments_model", expect_rows=False)

run_query(f'''
SELECT centroid_id, feature, ROUND(numerical_value, 2) AS value
FROM ML.CENTROIDS(MODEL {TBL("employee_segments_model")})
ORDER BY centroid_id, feature
''', label="Inspect the 3 cluster centroids")


In [ ]:

run_query(f'''
SELECT
  f.employee_id, f.department_name, f.avg_days_present, f.avg_leaves_taken,
  f.avg_performance_rating, p.centroid_id
FROM ML.PREDICT(MODEL {TBL("employee_segments_model")}, (
  SELECT employee_id, avg_days_present, avg_leaves_taken, avg_performance_rating
  FROM {TBL("employee_features_view")}
)) AS p
JOIN {TBL("employee_features_view")} AS f USING (employee_id)
ORDER BY p.centroid_id
LIMIT 15
''', label="Which cluster does each employee fall into?")



**What to look for:** compare the centroids against the per-employee assignments — one cluster
should show high attendance + high performance (your most reliable performers), another might show
higher leave-taking with lower performance, giving HR a data-driven starting point for segment-
specific engagement strategies instead of one-size-fits-all policy.

> 🖥️ **Check it in the console:** **hr_bqml_lab > employee_segments_model > Evaluation** tab shows
> a cluster visualization plotting the segments across two of the three features — a quick visual
> sanity check that the clusters are genuinely separated, not overlapping noise.



## Question 6 — Is Any Employee's Pattern Unusual Enough to Flag? (Anomaly Detection)

**The business problem:** rather than a human manually scanning 100 employees' attendance
patterns for outliers, `ML.DETECT_ANOMALIES` reuses the K-Means model from Question 5 to flag
employees whose behavior sits unusually far from *any* cluster centroid.

**How it works for K-Means specifically:** each employee's distance to their nearest centroid is
normalized by that cluster's typical spread: `contamination` sets what fraction of employees are
expected to be flagged (0.1 = roughly the most unusual 10%).


In [ ]:

run_query(f'''
SELECT
  employee_id, is_anomaly, ROUND(normalized_distance, 3) AS normalized_distance, centroid_id
FROM ML.DETECT_ANOMALIES(
  MODEL {TBL("employee_segments_model")},
  STRUCT(0.1 AS contamination),
  (SELECT employee_id, avg_days_present, avg_leaves_taken, avg_performance_rating
   FROM {TBL("employee_features_view")})
)
ORDER BY normalized_distance DESC
''', label="Anomaly detection on employee attendance patterns")



**What to look for:** the employees flagged `is_anomaly = TRUE` are exactly the ones whose
attendance pattern doesn't resemble any of the 3 typical clusters from Question 5 — a genuinely
useful, ranked starting point for an HR check-in, rather than a manual scan of every row.

> 🖥️ **Check it in the console:** anomaly detection results are query-only, same as Question 4's
> explanations — there's no separate anomaly-detection tab; the model itself is the same
> `employee_segments_model` visible in **hr_bqml_lab**.



## Question 7 — What Should Company-Wide Attendance Look Like Next Quarter? (Forecasting)

**The business problem:** workforce planning wants a data-driven expectation for the next few
months, not a guess — time series forecasting on the monthly attendance history.

**Honest caveat for the class:** 12 months of history is enough to demonstrate the *mechanism*,
but well short of the multiple full seasonal cycles `ARIMA_PLUS` really wants for a trustworthy
forecast — treat this as "how the tool works," not "trust this specific forecast."


In [ ]:

run_query(f'''
CREATE OR REPLACE MODEL {TBL("attendance_forecast_model")}
OPTIONS(
  model_type='ARIMA_PLUS',
  time_series_timestamp_col='month_date',
  time_series_data_col='avg_days_present',
  auto_arima=TRUE
) AS
SELECT month_date, AVG(days_present) AS avg_days_present
FROM {TBL("attendance_monthly")}
GROUP BY month_date
''', label="Train attendance_forecast_model", expect_rows=False)

run_query(f'''
SELECT * FROM ML.ARIMA_EVALUATE(MODEL {TBL("attendance_forecast_model")})
''', label="Evaluate candidate ARIMA models")


In [ ]:

run_query(f'''
SELECT
  forecast_timestamp, ROUND(forecast_value, 2) AS forecast_avg_days_present,
  ROUND(prediction_interval_lower_bound, 2) AS lower_bound,
  ROUND(prediction_interval_upper_bound, 2) AS upper_bound
FROM ML.FORECAST(MODEL {TBL("attendance_forecast_model")}, STRUCT(3 AS horizon, 0.95 AS confidence_level))
ORDER BY forecast_timestamp
''', label="3-month attendance forecast")



> 🖥️ **Check it in the console:** **hr_bqml_lab > attendance_forecast_model > Evaluation** tab
> plots the forecast with confidence-interval bands directly — a genuinely useful visual for
> explaining forecast uncertainty to workforce planning, without writing any charting code.



## Summary — the BQML Pattern, Restated

Every one of these seven questions followed the same three-step shape:

1. **`CREATE MODEL ... OPTIONS(model_type=...) AS SELECT ...`** to train
2. **`ML.EVALUATE`** to check quality (metric columns differ by model type)
3. **`ML.PREDICT`** (classification/regression/clustering), **`ML.FORECAST`** (time series), or
   **`ML.DETECT_ANOMALIES`** (unsupervised outlier flagging) to get results — plus
   **`ML.FEATURE_IMPORTANCE`**/**`ML.EXPLAIN_PREDICT`** for interpretability where supported

Six distinct capabilities, one dataset, zero data movement out of BigQuery, zero Python required.



## Cleanup

Deletes the dataset — every table, view, and all five models created in this notebook.


In [ ]:

CONFIRM_DELETE = False  # <-- set to True to actually delete resources

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [ ]:

if CONFIRM_DELETE:
    try:
        bq_client.delete_dataset(
            f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True
        )
        print(f"Deleted dataset {DATASET_ID} (every table, view, and all 5 models inside it).")
    except Exception as e:
        print("Cleanup skipped/failed:", e)

    print("\nCleanup complete.")



> 🖥️ **Final console check:** **BigQuery > SQL workspace > Explorer** → the `hr_bqml_lab` dataset
> should no longer be listed under your project at all, confirming every table, view, and model
> created in this notebook was removed.
